In [198]:
import random
import time
import math
import numpy as np

import deap
from deap import base, creator, tools

In [199]:
plantType = ["Null", "Corn", "Beans", "Squash"]

DEFAULT_NITROGEN = 50 # 50 ppm as baseline for the soil.
DEFAULT_LIGHT = 1 # 100% sunlight as baseline for the light level.
DEFAULT_YIELD = 0

CELLDISTANCE = 0

def create_plant(plantTypeVal):
    plantInd = [plantType[plantTypeVal], int(DEFAULT_NITROGEN), float(DEFAULT_LIGHT), float(DEFAULT_YIELD)]
    return plantInd

# Modifier Layout: [Plant-Corn Nitrogen, Plant-Corn Light, Plant-Beans Nitrogen, Plant-Beans Light, Plant-Squash Nitrogen, Plant-Squash Light]
CORN_MODIFIER = [-15, -.2, -10, .1, -5, -.25]
BEANS_MODIFIER = [20, 0, -5, -.1, 15, -.05]
SQUASH_MODIFIER = [5, 0, 5, -.05, -8, -.15]

def apply_plant_modifier(castingPlant, targetPlant):
    if (castingPlant[0] == "Null"):
        return targetPlant

    modifier = []
    if (castingPlant[0] == "Corn"):
        modifier = CORN_MODIFIER
    elif (castingPlant[0] == "Beans"):
        modifier = BEANS_MODIFIER
    elif (castingPlant[0] == "Squash"):
        modifier = SQUASH_MODIFIER

    if (targetPlant[0] == "Corn"):
        targetPlant[1] += modifier[0]
        targetPlant[2] += modifier[1]
    elif (targetPlant[0] == "Beans"):
        targetPlant[1] += modifier[2]
        targetPlant[2] += modifier[3]
    elif (targetPlant[0] == "Squash"):
        targetPlant[1] += modifier[4]
        targetPlant[2] += modifier[5]

    # Clamp nitrogen >= 0 and light to [0, 1]
    targetPlant[1] = max(0, targetPlant[1])
    targetPlant[2] = max(0.0, min(1.0, targetPlant[2]))

    return targetPlant

def calculate_all_modifiers(plantArray):
    for row in range(len(plantArray)):
        for col in range(len(plantArray[row])):
            curPlant = plantArray[row][col]
            if (row != 0):
                apply_plant_modifier(plantArray[row-1][col], curPlant)
                if (col != 0):
                    apply_plant_modifier(plantArray[row-1][col-1], curPlant)
                if (col != plantArray.shape[1]-1):
                    apply_plant_modifier(plantArray[row-1][col+1], curPlant)
            if (row != plantArray.shape[0]-1):
                apply_plant_modifier(plantArray[row+1][col], curPlant)
                if (col != 0):
                    apply_plant_modifier(plantArray[row+1][col-1], curPlant)
                if (col != plantArray.shape[1]-1):
                    apply_plant_modifier(plantArray[row+1][col+1], curPlant)
            if (col != 0):
                apply_plant_modifier(plantArray[row][col-1], curPlant)
            if (col != plantArray.shape[1]-1):
                apply_plant_modifier(plantArray[row][col+1], curPlant)

def calculate_yield_val(value, optMin, optMax):
    mid = (optMin + optMax) / 2
    half_range = (optMax - optMin) / 2
    yieldVal = max(0, half_range - abs(value - mid))
    return yieldVal

CORN_NITROGEN_RANGE = [120, 200]
CORN_LIGHT_RANGE = [.8, 1]
BEANS_NITROGEN_RANGE = [30, 60]
BEANS_LIGHT_RANGE = [.5, .75]
SQUASH_NITROGEN_RANGE = [60, 120]
SQUASH_LIGHT_RANGE = [.65, .9]

def calculate_plant_yield(plantInd):
    if (plantInd[0] == "Null"):
        return plantInd

    nitrogenRange = []
    lightRange = []

    if (plantInd[0] == "Corn"):
        nitrogenRange = CORN_NITROGEN_RANGE
        lightRange = CORN_LIGHT_RANGE
    elif (plantInd[0] == "Beans"):
        nitrogenRange = BEANS_NITROGEN_RANGE
        lightRange = BEANS_LIGHT_RANGE
    elif (plantInd[0] == "Squash"):
        nitrogenRange = SQUASH_NITROGEN_RANGE
        lightRange = SQUASH_LIGHT_RANGE

    nitrogenValue = calculate_yield_val(plantInd[1], nitrogenRange[0], nitrogenRange[1])
    lightValue = calculate_yield_val(plantInd[2], lightRange[0], lightRange[1])

    plantInd[3] = nitrogenValue * lightValue
    return plantInd

def calculate_all_yields(plantArray):
    totalYield = 0
    for row in range(len(plantArray)):
        for col in range(len(plantArray[row])):
            calculate_plant_yield(plantArray[row][col])
            totalYield += plantArray[row][col][3]
    return totalYield


In [ ]:
# Test functions

def print_plant_array(plantArray):
    for row in plantArray:
        print("[ ", end="")
        for i in row:
            print(f"[T: {i[0]}, Y: {i[3]}] ", end="")
        print("],")

def print_full_plant_array(plantArray):
    for row in plantArray:
        print("[ ", end="")
        for i in row:
            print(f"[T: {i[0]}, N: {i[1]}, L: {i[2]}, Y: {i[3]}] ", end="")
        print("],")

def test_yields(xSize, ySize):
    fieldArray = np.array([[create_plant(random.randint(0, 3)) for _ in range(ySize)] for _ in range(xSize)], dtype=object)
    calculate_all_modifiers(fieldArray)
    totalYield = calculate_all_yields(fieldArray)
    print_full_plant_array(fieldArray)
    print(f"Total Yield: {totalYield}")

test_yields(5, 5)
    

[ [T: Corn, N: 95, L: 1.0, Y: 0] [T: Squash, N: 70, L: 0.3999999999999999, Y: 0.0] [T: Corn, N: 80, L: 1.0, Y: 0] [T: Squash, N: 75, L: 0.6499999999999999, Y: 0.0] [T: Beans, N: 50, L: 0.85, Y: 0.0] ],
[ [T: Beans, N: 30, L: 0.9, Y: 0] [T: Beans, N: 20, L: 0.9, Y: 0] [T: Null, N: 50, L: 1.0, Y: 0.0] [T: Null, N: 50, L: 1.0, Y: 0.0] [T: Beans, N: 40, L: 0.65, Y: 0.9999999999999998] ],
[ [T: Null, N: 50, L: 1.0, Y: 0.0] [T: Corn, N: 95, L: 0.8, Y: 0] [T: Null, N: 50, L: 1.0, Y: 0.0] [T: Beans, N: 45, L: 0.75, Y: 0.0] [T: Beans, N: 45, L: 0.75, Y: 0.0] ],
[ [T: Beans, N: 25, L: 1.0, Y: 0] [T: Corn, N: 45, L: 0.6000000000000001, Y: 0] [T: Null, N: 50, L: 1.0, Y: 0.0] [T: Squash, N: 72, L: 0.7499999999999999, Y: 1.1999999999999984] [T: Null, N: 50, L: 1.0, Y: 0.0] ],
[ [T: Corn, N: 60, L: 0.8, Y: 0] [T: Squash, N: 55, L: 0.44999999999999996, Y: 0] [T: Null, N: 50, L: 1.0, Y: 0.0] [T: Null, N: 50, L: 1.0, Y: 0.0] [T: Squash, N: 42, L: 0.85, Y: 0.0] ],
Total Yield: 2.1999999999999984


In [ ]:
GRID_X = 5
GRID_Y = 5

def evaluate(individual):
    plantArray = np.array([[create_plant(individual[row * GRID_Y + col]) for col in range(GRID_Y)] for row in range(GRID_X)], dtype=object)
    calculate_all_modifiers(plantArray)
    totalYield = calculate_all_yields(plantArray)
    return (totalYield,)

def main():
    if not hasattr(creator, "FitnessMax"):
        creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    if not hasattr(creator, "Individual"):
        creator.create("Individual", list, fitness=creator.FitnessMax)

    toolbox = base.Toolbox()

    toolbox.register("attr_plant", create_plant, random.randint, 0, 3)
    toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.attr_plant)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)

    toolbox.register("evaluate", evaluate)
    toolbox.register("mate", tools.cxTwoPoint)
    toolbox.register("mutate", tools.mutUniformInt, low=0, up=3, indpb=0.1)
    toolbox.register("select", tools.selTournament, tournsize=3)
    
    POP_SIZE = 100
    N_GEN = 100
    CROSSOVER = 0.7
    MUTATION = 0.2

    pop = toolbox.population(n=POP_SIZE)
    hof = tools.HallOfFame(1)

    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("max", np.max)

# continue from here